# Financial Fraud Detection System

## Qwen2.5 QLoRA Fine-Tuning

This notebook performs supervised fine-tuning of
`Qwen/Qwen2.5-1.5B-Instruct` for financial fraud-risk
classification using QLoRA.

The notebook uses the reusable project modules already implemented in
`ml/src/`.

### Training workflow

1. Configure the Colab GPU environment
2. Clone or locate the project repository
3. Install ML dependencies
4. Load the real fraud dataset
5. Validate and preprocess transactions
6. Create leakage-safe train/test partitions
7. Balance the training partition
8. Convert transactions to Qwen conversations
9. Load Qwen in 4-bit NF4
10. Attach LoRA adapters
11. Run supervised fine-tuning
12. Save the trained adapter and tokenizer
13. Save training metadata and training history

The test partition is never used for model optimization.


## 1. GPU Environment Check

QLoRA requires a CUDA-capable GPU for the project's 4-bit
`bitsandbytes` configuration.

In Google Colab select:

**Runtime → Change runtime type → T4 GPU**

before running the training cells.


In [23]:
%cd /content/Financial-Fraud-Detection-System

!git pull origin main
!git log -1 --oneline

/content/Financial-Fraud-Detection-System
From https://github.com/Ebbeju-Lankapalli/Financial-Fraud-Detection-System
 * branch            main       -> FETCH_HEAD
Already up to date.
0ebb504 (HEAD -> main, origin/main, origin/HEAD) Created using Colab


In [16]:
%cd /content/Financial-Fraud-Detection-System

!git pull origin main
!git log -1 --oneline

/content/Financial-Fraud-Detection-System
From https://github.com/Ebbeju-Lankapalli/Financial-Fraud-Detection-System
 * branch            main       -> FETCH_HEAD
Already up to date.
0ebb504 (HEAD -> main, origin/main, origin/HEAD) Created using Colab


In [17]:
import gc
import importlib
import shutil
import torch

import ml.src.training.train as train_module

importlib.reload(train_module)

from ml.src.training.train import (
    load_quantized_model,
    prepare_lora_model,
    train_model,
)

gc.collect()
torch.cuda.empty_cache()

shutil.rmtree(
    "/content/Financial-Fraud-Detection-System/artifacts/fraud-qlora-adapter",
    ignore_errors=True,
)

print("Updated training module loaded.")
print("Old incomplete adapter directory removed.")
print("GPU:", torch.cuda.get_device_name(0))

Updated training module loaded.
Old incomplete adapter directory removed.
GPU: Tesla T4


In [18]:
from collections import Counter

test_model = load_quantized_model(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

test_model = prepare_lora_model(
    test_model
)

trainable_dtypes = Counter(
    str(parameter.dtype)
    for parameter in test_model.parameters()
    if parameter.requires_grad
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in test_model.parameters()
    if parameter.requires_grad
)

print("=" * 70)
print("QLORA DTYPE VERIFICATION")
print("=" * 70)

print("Trainable parameter dtypes:", trainable_dtypes)
print("Trainable parameters:", f"{trainable_parameters:,}")

assert set(trainable_dtypes) == {"torch.float32"}

print()
print("All trainable LoRA parameters are FP32: PASS")
print("=" * 70)

del test_model
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

QLORA DTYPE VERIFICATION
Trainable parameter dtypes: Counter({'torch.float32': 224})
Trainable parameters: 4,358,144

All trainable LoRA parameters are FP32: PASS


In [1]:
import platform

print("Python environment:", platform.python_version())

try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(
            "CUDA version:",
            torch.version.cuda,
        )
    else:
        print(
            "WARNING: CUDA GPU not detected. "
            "Do not start QLoRA training."
        )

except ImportError:
    print(
        "PyTorch is not installed yet. "
        "It will be installed in the dependency step."
    )


Python environment: 3.12.13
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


## 2. Project Setup

The training notebook expects the complete Financial Fraud Detection
System repository.

When running in Colab, clone the GitHub repository after the project has
been pushed to GitHub.

If the repository is already available in the runtime, skip the clone
command and set `PROJECT_ROOT` to the existing repository directory.


In [2]:
from pathlib import Path

COLAB_ROOT = Path("/content")

PROJECT_NAME = "Financial-Fraud-Detection-System"

PROJECT_ROOT = COLAB_ROOT / PROJECT_NAME

print("Expected project root:")
print(PROJECT_ROOT)


Expected project root:
/content/Financial-Fraud-Detection-System


### Repository clone

Before executing this cell in Colab, replace `YOUR_GITHUB_REPOSITORY_URL`
with the repository URL after the project has been pushed to GitHub.

Do not execute the placeholder command unchanged.


In [8]:
!git clone https://github.com/Ebbeju-Lankapalli/Financial-Fraud-Detection-System.git /content/Financial-Fraud-Detection-System
%cd /content/Financial-Fraud-Detection-System
!git log -1 --oneline

Cloning into '/content/Financial-Fraud-Detection-System'...
remote: Enumerating objects: 233, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 233 (delta 48), reused 224 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (233/233), 178.70 KiB | 8.93 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/Financial-Fraud-Detection-System
0ebb504 (HEAD -> main, origin/main, origin/HEAD) Created using Colab


## 3. Install Training Dependencies

The project pins the ML stack in `ml/requirements.txt`.

The notebook installs those exact project dependencies so the Colab
training environment matches the repository configuration.


In [1]:
%cd /content/Financial-Fraud-Detection-System
!pip install -r ml/requirements.txt

/content/Financial-Fraud-Detection-System


In [2]:
!pip uninstall -y torchvision torchaudio

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [4]:
import torch
import accelerate
import bitsandbytes
import datasets
import peft
import transformers
import trl

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

print("DEPENDENCY CHECK PASSED")

PyTorch: 2.13.0+cu130
CUDA: 13.0
CUDA available: True
GPU: Tesla T4
Transformers: 5.15.0
Datasets: 5.0.1
PEFT: 0.20.0
TRL: 1.9.2
Accelerate: 1.14.0
BitsAndBytes: 0.50.0
DEPENDENCY CHECK PASSED


In [3]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/Financial-Fraud-Detection-System")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current directory:", os.getcwd())

Current directory: /content/Financial-Fraud-Detection-System


In [5]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/Financial-Fraud-Detection-System")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current directory:", os.getcwd())
print("Project root in sys.path:", str(PROJECT_ROOT) in sys.path)

Current directory: /content/Financial-Fraud-Detection-System
Project root in sys.path: True


In [4]:
# Run this after the repository has been cloned in Colab:
#
# %cd /content/Financial-Fraud-Detection-System
# !pip install -r ml/requirements.txt


### Runtime restart

If Colab requests a runtime restart after installing PyTorch,
Transformers, TRL, PEFT, or bitsandbytes, restart the runtime and then
continue from the environment verification section below.


In [5]:
# Dependency verification cell.
# Run after installing ml/requirements.txt.

try:
    import accelerate
    import bitsandbytes
    import datasets
    import peft
    import transformers
    import trl

    print("Transformers:", transformers.__version__)
    print("Datasets:", datasets.__version__)
    print("PEFT:", peft.__version__)
    print("TRL:", trl.__version__)
    print("Accelerate:", accelerate.__version__)
    print("bitsandbytes:", bitsandbytes.__version__)

except ImportError as exc:
    print(
        "Dependencies are not fully installed yet:",
        exc,
    )


Dependencies are not fully installed yet: No module named 'bitsandbytes'


## 4. Import Project Pipeline

The notebook does not duplicate the project's ML implementation.

Instead, it imports the reusable loading, preprocessing, splitting,
conversation-formatting, and training modules from `ml/src/`.


In [6]:
from ml.src.data.conversation_format import (
    convert_dataset_to_conversations,
)
from ml.src.data.load_data import (
    load_fraud_dataset,
    select_working_subset,
)
from ml.src.data.preprocess import (
    preprocess_dataset,
    validate_dataset_schema,
)
from ml.src.data.split_data import (
    get_class_counts,
    prepare_balanced_splits,
)
from ml.src.training.train import (
    DEFAULT_MODEL_CONFIG_PATH,
    DEFAULT_TRAINING_CONFIG_PATH,
    get_base_model_id,
    load_yaml_config,
    train_model,
)
from ml.src.utils.constants import (
    DATASET_ID,
    RANDOM_SEED,
    TEST_DATASET_SIZE,
    TRAIN_DATASET_SIZE,
    WORKING_SUBSET_SIZE,
)
from ml.src.utils.seed import set_global_seed

set_global_seed(RANDOM_SEED)

print("Project ML pipeline imported successfully.")


Project ML pipeline imported successfully.


## 5. Confirm Frozen Training Configuration

The project configuration is loaded directly from the YAML files committed
to the repository.

This keeps notebook execution consistent with the reusable training engine.


In [7]:
model_config = load_yaml_config(
    DEFAULT_MODEL_CONFIG_PATH
)

training_config = load_yaml_config(
    DEFAULT_TRAINING_CONFIG_PATH
)

base_model_id = get_base_model_id(
    model_config
)

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Dataset:", DATASET_ID)
print("Base model:", base_model_id)
print("Random seed:", RANDOM_SEED)
print("Working subset:", WORKING_SUBSET_SIZE)
print("Training rows:", TRAIN_DATASET_SIZE)
print("Test rows:", TEST_DATASET_SIZE)

print(
    "Epochs:",
    training_config["training"]["num_train_epochs"],
)

print(
    "Learning rate:",
    training_config["training"]["learning_rate"],
)

print(
    "Batch size:",
    training_config["training"][
        "per_device_train_batch_size"
    ],
)

print(
    "Gradient accumulation:",
    training_config["training"][
        "gradient_accumulation_steps"
    ],
)

print("=" * 70)


TRAINING CONFIGURATION
Dataset: CiferAI/Cifer-Fraud-Detection-Dataset-AF
Base model: Qwen/Qwen2.5-1.5B-Instruct
Random seed: 42
Working subset: 1000000
Training rows: 2000
Test rows: 500
Epochs: 3
Learning rate: 0.0002
Batch size: 4
Gradient accumulation: 2


## 6. Load the Real Fraud Dataset

The source dataset is:

`CiferAI/Cifer-Fraud-Detection-Dataset-AF`

The project first selects its configured working subset before
preprocessing and model-specific sampling.


In [8]:
raw_dataset = load_fraud_dataset()

print("Dataset loaded successfully.")
print("Available rows:", len(raw_dataset))
print("Columns:", raw_dataset.column_names)


README.md:   0%|          | 0.00/5.07k [00:00<?, ?B/s]

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): reconstructing file:   0%|          |  0.00B /  138MB            

Cifer-Fraud-Detection-Dataset-AF-part-1-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-10(…): downloading bytes:           |  0.00B            

...

Cifer-Fraud-Detection-Dataset-AF-part-8-(…): downloading bytes:           |  0.00B            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): reconstructing file:   0%|          |  0.00B /  127MB            

Cifer-Fraud-Detection-Dataset-AF-part-9-(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/21000000 [00:00<?, ? examples/s]

Dataset loaded successfully.
Available rows: 21000000
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [9]:
working_dataset = select_working_subset(
    raw_dataset,
    subset_size=WORKING_SUBSET_SIZE,
    seed=RANDOM_SEED,
)

print("Working rows:", len(working_dataset))


Working rows: 1000000


## 7. Validate and Preprocess Transactions

The preprocessing pipeline:

- verifies required model features
- removes invalid transactions
- normalizes transaction values
- keeps only the features required by the fraud model

`isFlaggedFraud` is intentionally excluded from model inputs because it
could act as a target proxy.


In [10]:
validate_dataset_schema(
    working_dataset
)

clean_dataset = preprocess_dataset(
    working_dataset
)

print("Preprocessed rows:", len(clean_dataset))
print("Model columns:", clean_dataset.column_names)
print("Class counts:", get_class_counts(clean_dataset))


Filtering invalid transactions:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Normalizing transactions:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Preprocessed rows: 1000000
Model columns: ['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud']
Class counts: {0: 998700, 1: 1300}


## 8. Leakage-Safe Train/Test Preparation

The project separates train and test data before balancing.

This is important because balancing before splitting could allow duplicated
or related sampled observations to contaminate model evaluation.

Target sizes:

- Training: 2,000 transactions
- Testing: 500 transactions

Both partitions are balanced independently.


In [11]:
prepared = prepare_balanced_splits(
    dataset=clean_dataset,
    train_samples_per_class=(
        TRAIN_DATASET_SIZE // 2
    ),
    test_samples_per_class=(
        TEST_DATASET_SIZE // 2
    ),
    seed=RANDOM_SEED,
)

train_dataset = prepared["train"]
test_dataset = prepared["test"]

print("=" * 70)
print("PREPARED DATA")
print("=" * 70)

print(
    "Training:",
    len(train_dataset),
    get_class_counts(train_dataset),
)

print(
    "Testing:",
    len(test_dataset),
    get_class_counts(test_dataset),
)

print("=" * 70)


Selecting class 0:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Selecting class 1:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Selecting class 0:   0%|          | 0/800000 [00:00<?, ? examples/s]

Selecting class 1:   0%|          | 0/800000 [00:00<?, ? examples/s]

Selecting class 0:   0%|          | 0/200000 [00:00<?, ? examples/s]

Selecting class 1:   0%|          | 0/200000 [00:00<?, ? examples/s]

PREPARED DATA
Training: 2000 {0: 1000, 1: 1000}
Testing: 500 {0: 250, 1: 250}


## 9. Verify Train/Test Separation

The test partition must remain completely independent from the training
partition.

Evaluation will use this test set only after fine-tuning has completed.


In [12]:
print("Training rows:", len(train_dataset))
print("Testing rows:", len(test_dataset))

print(
    "Training class counts:",
    get_class_counts(train_dataset),
)

print(
    "Testing class counts:",
    get_class_counts(test_dataset),
)

assert len(train_dataset) == TRAIN_DATASET_SIZE
assert len(test_dataset) == TEST_DATASET_SIZE

print("Dataset size checks passed.")


Training rows: 2000
Testing rows: 500
Training class counts: {0: 1000, 1: 1000}
Testing class counts: {0: 250, 1: 250}
Dataset size checks passed.


## 10. Preview Qwen Training Conversations

Each transaction becomes a supervised chat conversation.

The user message contains the transaction information and the assistant
target contains the fraud-risk class expected during fine-tuning.


In [13]:
conversation_preview = (
    convert_dataset_to_conversations(
        train_dataset.select(
            range(min(3, len(train_dataset)))
        )
    )
)

for index in range(
    len(conversation_preview)
):
    print("=" * 70)
    print("EXAMPLE", index + 1)
    print("=" * 70)

    for message in (
        conversation_preview[index]["messages"]
    ):
        print(
            message["role"].upper() + ":"
        )
        print(message["content"])
        print()


Creating training conversations:   0%|          | 0/3 [00:00<?, ? examples/s]

EXAMPLE 1
USER:
Analyze this transaction for fraud risk:
- Type: PAYMENT
- Amount: $157,360.78
- Sender Balance Before: $1,959.49
- Sender Balance After: $3,591.92
- Recipient Balance Before: $1,081,906.74
- Recipient Balance After: $540.10

ASSISTANT:
LOW

EXAMPLE 2
USER:
Analyze this transaction for fraud risk:
- Type: TRANSFER
- Amount: $72,318.69
- Sender Balance Before: $3,720,428.43
- Sender Balance After: $2,382,148.67
- Recipient Balance Before: $106,686.77
- Recipient Balance After: $22,419.37

ASSISTANT:
HIGH

EXAMPLE 3
USER:
Analyze this transaction for fraud risk:
- Type: PAYMENT
- Amount: $520,239.87
- Sender Balance Before: $67,537.27
- Sender Balance After: $321,936.47
- Recipient Balance Before: $544,805.33
- Recipient Balance After: $798,900.89

ASSISTANT:
LOW



## 11. Final GPU Safety Check

The next section performs actual QLoRA fine-tuning.

Do not continue unless CUDA is available.


In [14]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for the configured "
        "4-bit QLoRA training pipeline."
    )

print("CUDA available:", torch.cuda.is_available())
print("Training GPU:", torch.cuda.get_device_name(0))

gpu_properties = torch.cuda.get_device_properties(0)

print(
    "GPU memory:",
    round(
        gpu_properties.total_memory
        / (1024 ** 3),
        2,
    ),
    "GB",
)

print("GPU safety check passed.")


CUDA available: True
Training GPU: Tesla T4
GPU memory: 14.56 GB
GPU safety check passed.


## 12. QLoRA Fine-Tuning

This is the computationally expensive training cell.

The reusable `train_model()` pipeline will:

1. load Qwen2.5-1.5B-Instruct
2. apply 4-bit NF4 quantization
3. prepare the model for k-bit training
4. attach LoRA adapters
5. render the training conversations
6. create the TRL SFT trainer
7. train for the configured epochs
8. save the LoRA adapter
9. save the tokenizer
10. save reproducibility metadata

The independent test partition is not passed to the trainer.


In [20]:
%cd /content/Financial-Fraud-Detection-System

!git pull origin main
!git log -1 --oneline

/content/Financial-Fraud-Detection-System
From https://github.com/Ebbeju-Lankapalli/Financial-Fraud-Detection-System
 * branch            main       -> FETCH_HEAD
Already up to date.
0ebb504 (HEAD -> main, origin/main, origin/HEAD) Created using Colab


In [25]:
from accelerate.state import AcceleratorState
from ml.src.training.train import (
    DEFAULT_TRAINING_CONFIG_PATH,
    build_training_arguments,
    load_yaml_config,
)

config = load_yaml_config(
    DEFAULT_TRAINING_CONFIG_PATH
)

args = build_training_arguments(
    config,
    "/content/fraud-check",
    training_rows=2000,
)

print("=" * 60)
print("LIVE PRECISION STATE")
print("=" * 60)

print("SFTConfig fp16:", args.fp16)
print("SFTConfig bf16:", args.bf16)

try:
    state = AcceleratorState()
    print(
        "Accelerate mixed precision:",
        state.mixed_precision,
    )
except Exception as exc:
    print(
        "Accelerate state not initialized:",
        type(exc).__name__,
        exc,
    )

print("=" * 60)

LIVE PRECISION STATE
SFTConfig fp16: True
SFTConfig bf16: False
Accelerate state not initialized: ValueError Please make sure to properly initialize your accelerator via `accelerator = Accelerator()` before using any functionality from the `accelerate` library.


In [26]:
%cd /content/Financial-Fraud-Detection-System

!git pull origin main
!git log -1 --oneline

/content/Financial-Fraud-Detection-System
From https://github.com/Ebbeju-Lankapalli/Financial-Fraud-Detection-System
 * branch            main       -> FETCH_HEAD
Already up to date.
0ebb504 (HEAD -> main, origin/main, origin/HEAD) Created using Colab


In [27]:
!grep -n -A5 -B2 "precision:" ml/configs/training_config.yaml

48-# ============================================================
49-
50:precision:
51-  fp16: true
52-
53-  bf16: false
54-
55-


In [30]:
%cd /content/Financial-Fraud-Detection-System

!git pull origin main
!git log -1 --oneline

!grep -n -A5 -B2 "precision:" ml/configs/training_config.yaml

/content/Financial-Fraud-Detection-System
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 11 (delta 8), reused 11 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 1.15 KiB | 294.00 KiB/s, done.
From https://github.com/Ebbeju-Lankapalli/Financial-Fraud-Detection-System
 * branch            main       -> FETCH_HEAD
   0ebb504..f57eb76  main       -> origin/main
Updating 0ebb504..f57eb76
Fast-forward
 ml/configs/training_config.yaml |  2 +-
 ml/src/training/train.py        | 11 +++++++++++
 2 files changed, 12 insertions(+), 1 deletion(-)
f57eb76 (HEAD -> main, origin/main, origin/HEAD) fix: disable trainer AMP for stable T4 QLoRA
48-# ============================================================
49-
50:precision:
51-  fp16: false
52-
53-  bf16: false
54-
55-


In [31]:
from pathlib import Path

OUTPUT_DIR = Path(
    "artifacts/fraud-qlora-adapter"
)

print("Adapter output:", OUTPUT_DIR.resolve())
print("Training examples:", len(train_dataset))

trainer = train_model(
    train_dataset=train_dataset,
    output_dir=OUTPUT_DIR,
)

print("QLoRA fine-tuning completed.")


Adapter output: /content/Financial-Fraud-Detection-System/artifacts/fraud-qlora-adapter
Training examples: 2000


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.701725
20,1.892210
30,1.050342
40,0.833669
50,0.749964
60,0.732847
70,0.728389
80,0.727955
90,0.742987
100,0.718976


QLoRA fine-tuning completed.


## 13. Training Results

After training, inspect the trainer history.

The loss values generated here will later be used by the evaluation
pipeline and project visualizations.


In [32]:
training_history = trainer.state.log_history

print("Training history entries:", len(training_history))

for entry in training_history:
    print(entry)


Training history entries: 76
{'loss': 2.701724815368652, 'grad_norm': 2.046875, 'learning_rate': 8.181818181818183e-05, 'entropy': 1.6999574959278108, 'num_tokens': 10358.0, 'mean_token_accuracy': 0.44371199756860735, 'epoch': 0.04, 'step': 10}
{'loss': 1.8922103881835937, 'grad_norm': 1.2109375, 'learning_rate': 0.00017272727272727275, 'entropy': 1.7188256859779358, 'num_tokens': 20728.0, 'mean_token_accuracy': 0.5372154146432877, 'epoch': 0.08, 'step': 20}
{'loss': 1.0503421783447267, 'grad_norm': 0.5625, 'learning_rate': 0.00019995437844895334, 'entropy': 1.1111923038959504, 'num_tokens': 30997.0, 'mean_token_accuracy': 0.6784297704696656, 'epoch': 0.12, 'step': 30}
{'loss': 0.8336685180664063, 'grad_norm': 0.46484375, 'learning_rate': 0.00019973102615702422, 'entropy': 0.8192181199789047, 'num_tokens': 41272.0, 'mean_token_accuracy': 0.7161162793636322, 'epoch': 0.16, 'step': 40}
{'loss': 0.7499642372131348, 'grad_norm': 0.1787109375, 'learning_rate': 0.00019932197900778537, 'entro

## 14. Save Training History

The adapter directory already contains the model adapter, tokenizer, and
training metadata.

We additionally save the trainer log history as JSON so training curves can
be generated reproducibly during evaluation.


In [33]:
import json

history_path = (
    OUTPUT_DIR
    / "training_history.json"
)

with history_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        training_history,
        file,
        indent=2,
    )

print(
    "Training history saved:",
    history_path.resolve(),
)


Training history saved: /content/Financial-Fraud-Detection-System/artifacts/fraud-qlora-adapter/training_history.json


## 15. Verify Saved Adapter

The adapter directory should contain the LoRA adapter configuration and
weights together with tokenizer and project metadata.

These artifacts are much smaller than storing another complete copy of the
base Qwen model.


In [34]:
print("Saved training artifacts:")

for path in sorted(
    OUTPUT_DIR.rglob("*")
):
    if path.is_file():
        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"{path} "
            f"({size_mb:.2f} MB)"
        )


Saved training artifacts:
artifacts/fraud-qlora-adapter/README.md (0.00 MB)
artifacts/fraud-qlora-adapter/adapter_config.json (0.00 MB)
artifacts/fraud-qlora-adapter/adapter_model.safetensors (8.34 MB)
artifacts/fraud-qlora-adapter/chat_template.jinja (0.00 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/README.md (0.00 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/adapter_config.json (0.00 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/adapter_model.safetensors (8.34 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/chat_template.jinja (0.00 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/optimizer.pt (8.70 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/rng_state.pth (0.01 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/scheduler.pt (0.00 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/tokenizer.json (10.89 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/tokenizer_config.json (0.00 MB)
artifacts/fraud-qlora-adapter/checkpoint-250/trainer_state.json (0.01 MB)
artifacts/fraud

## 16. Preserve the Independent Test Dataset

The test partition will be used in the next project stage to compare:

- base Qwen model performance
- fine-tuned fraud model performance

It must not be used for gradient updates or training decisions.


In [35]:
print("=" * 70)
print("TRAINING STAGE COMPLETE")
print("=" * 70)

print("Base model:", base_model_id)
print("Training method: QLoRA")
print("Training rows:", len(train_dataset))
print("Held-out test rows:", len(test_dataset))
print("Adapter directory:", OUTPUT_DIR.resolve())

print()
print(
    "The held-out test dataset remains reserved "
    "for model evaluation."
)

print("=" * 70)


TRAINING STAGE COMPLETE
Base model: Qwen/Qwen2.5-1.5B-Instruct
Training method: QLoRA
Training rows: 2000
Held-out test rows: 500
Adapter directory: /content/Financial-Fraud-Detection-System/artifacts/fraud-qlora-adapter

The held-out test dataset remains reserved for model evaluation.


## Next Stage

The next notebook will perform model evaluation.

It will compare the original base model with the QLoRA fine-tuned model on
the independent fraud test partition using classification metrics such as:

- accuracy
- precision
- recall
- F1-score
- confusion matrix

The comparison will demonstrate whether domain-specific fine-tuning
improved financial fraud-risk classification.


In [36]:
from huggingface_hub import notebook_login

notebook_login()

In [37]:
from huggingface_hub import HfApi

api = HfApi()

account = api.whoami()

print("=" * 60)
print("HUGGING FACE AUTHENTICATION")
print("=" * 60)
print("Username:", account["name"])
print("Authentication: SUCCESS")
print("=" * 60)

HUGGING FACE AUTHENTICATION
Username: ebbejulankapalli
Authentication: SUCCESS


In [38]:
from huggingface_hub import HfApi

api = HfApi()

REPO_ID = "ebbejulankapalli/financial-fraud-detector-qwen2.5-qlora"

repo_url = api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)

print("=" * 70)
print("HUGGING FACE MODEL REPOSITORY")
print("=" * 70)
print("Repository:", REPO_ID)
print("URL:", repo_url)
print("Repository creation: SUCCESS")
print("=" * 70)

HUGGING FACE MODEL REPOSITORY
Repository: ebbejulankapalli/financial-fraud-detector-qwen2.5-qlora
URL: https://huggingface.co/ebbejulankapalli/financial-fraud-detector-qwen2.5-qlora
Repository creation: SUCCESS


In [39]:
from huggingface_hub import HfApi

api = HfApi()

REPO_ID = "ebbejulankapalli/financial-fraud-detector-qwen2.5-qlora"

ADAPTER_DIR = (
    "/content/Financial-Fraud-Detection-System/"
    "artifacts/fraud-qlora-adapter"
)

api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=ADAPTER_DIR,
    ignore_patterns=[
        "checkpoint-*",
        "checkpoint-*/**",
    ],
)

print("=" * 70)
print("QLORA ADAPTER UPLOAD COMPLETE")
print("=" * 70)
print("Repository:", REPO_ID)
print("Uploaded final adapter artifacts only")
print("Checkpoint folders skipped")
print("=" * 70)

QLORA ADAPTER UPLOAD COMPLETE
Repository: ebbejulankapalli/financial-fraud-detector-qwen2.5-qlora
Uploaded final adapter artifacts only
Checkpoint folders skipped


In [40]:
from huggingface_hub import HfApi

api = HfApi()

REPO_ID = "ebbejulankapalli/financial-fraud-detector-qwen2.5-qlora"

files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model",
)

print("=" * 70)
print("HUGGING FACE ADAPTER VERIFICATION")
print("=" * 70)

for file in files:
    print(file)

required_files = {
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "training_metadata.json",
    "training_history.json",
}

missing = required_files - set(files)

assert not missing, f"Missing files: {sorted(missing)}"

print()
print("Required adapter files: PASS")
print("Remote QLoRA adapter: READY")
print("=" * 70)

HUGGING FACE ADAPTER VERIFICATION
.gitattributes
README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
tokenizer.json
tokenizer_config.json
training_args.bin
training_history.json
training_metadata.json

Required adapter files: PASS
Remote QLoRA adapter: READY


In [42]:
from huggingface_hub import snapshot_download

ADAPTER_REPO_ID = (
    "ebbejulankapalli/"
    "financial-fraud-detector-qwen2.5-qlora"
)

LOCAL_ADAPTER_DIR = snapshot_download(
    repo_id=ADAPTER_REPO_ID,
    repo_type="model",
)

print("Downloaded adapter to:")
print(LOCAL_ADAPTER_DIR)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded adapter to:
/root/.cache/huggingface/hub/models--ebbejulankapalli--financial-fraud-detector-qwen2.5-qlora/snapshots/5729224c4044daca36f1fed61dfb37cce7bd36c0


In [43]:
import gc
import torch

from ml.src.training.train import (
    load_adapter_for_inference,
)

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = load_adapter_for_inference(
    LOCAL_ADAPTER_DIR
)

print("=" * 70)
print("FINE-TUNED MODEL LOADED")
print("=" * 70)
print("Adapter:", ADAPTER_REPO_ID)
print("Local adapter:", LOCAL_ADAPTER_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("Held-out test rows:", len(test_dataset))
print("=" * 70)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

FINE-TUNED MODEL LOADED
Adapter: ebbejulankapalli/financial-fraud-detector-qwen2.5-qlora
Local adapter: /root/.cache/huggingface/hub/models--ebbejulankapalli--financial-fraud-detector-qwen2.5-qlora/snapshots/5729224c4044daca36f1fed61dfb37cce7bd36c0
GPU: Tesla T4
Held-out test rows: 500


In [45]:
import time

import torch
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from ml.src.data.conversation_format import create_conversation


def parse_prediction(response: str):
    """Extract HIGH/LOW from generated model text."""

    normalized = response.strip().upper()

    if normalized.startswith("HIGH"):
        return "HIGH"

    if normalized.startswith("LOW"):
        return "LOW"

    return None


def predict_transaction(
    model,
    tokenizer,
    row,
):
    """Predict fraud risk for one held-out transaction."""

    conversation = create_conversation(row)

    # IMPORTANT:
    # Only give the USER message to the model.
    # Never expose the true assistant HIGH/LOW label.
    messages = [
        conversation["messages"][0]
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    # Move every tensor to the model's device.
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0,
        prompt_length:,
    ]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    prediction = parse_prediction(response)

    return prediction, response


# ============================================================
# Evaluation
# ============================================================

actual_labels = []
predicted_labels = []
raw_responses = []
invalid_predictions = []

start_time = time.time()

print("=" * 70)
print("FINE-TUNED MODEL — HELD-OUT EVALUATION")
print("=" * 70)
print("Test transactions:", len(test_dataset))
print()


for index, row in enumerate(test_dataset):

    actual = (
        "HIGH"
        if int(row["isFraud"]) == 1
        else "LOW"
    )

    prediction, raw_response = predict_transaction(
        model,
        tokenizer,
        row,
    )

    actual_labels.append(actual)
    raw_responses.append(raw_response)

    if prediction is None:

        predicted_labels.append("INVALID")

        invalid_predictions.append(
            {
                "index": index,
                "actual": actual,
                "response": raw_response,
            }
        )

    else:
        predicted_labels.append(prediction)

    if (index + 1) % 50 == 0:

        elapsed = time.time() - start_time

        print(
            f"Processed {index + 1}/"
            f"{len(test_dataset)} "
            f"transactions "
            f"({elapsed:.1f}s)"
        )


# ============================================================
# Metrics
# ============================================================

accuracy = accuracy_score(
    actual_labels,
    predicted_labels,
)

precision = precision_score(
    actual_labels,
    predicted_labels,
    pos_label="HIGH",
    zero_division=0,
)

recall = recall_score(
    actual_labels,
    predicted_labels,
    pos_label="HIGH",
    zero_division=0,
)

f1 = f1_score(
    actual_labels,
    predicted_labels,
    pos_label="HIGH",
    zero_division=0,
)

cm = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=["LOW", "HIGH"],
)

tn, fp, fn, tp = cm.ravel()

elapsed = time.time() - start_time


# ============================================================
# Results
# ============================================================

print()
print("=" * 70)
print("FINE-TUNED MODEL RESULTS")
print("=" * 70)

print(
    f"Accuracy:  {accuracy:.4f} "
    f"({accuracy:.2%})"
)

print(
    f"Precision: {precision:.4f} "
    f"({precision:.2%})"
)

print(
    f"Recall:    {recall:.4f} "
    f"({recall:.2%})"
)

print(
    f"F1 Score:  {f1:.4f} "
    f"({f1:.2%})"
)

print()

print("Confusion Matrix")
print("----------------")

print(
    f"True Negatives  (LOW  → LOW):  {tn}"
)

print(
    f"False Positives (LOW  → HIGH): {fp}"
)

print(
    f"False Negatives (HIGH → LOW):  {fn}"
)

print(
    f"True Positives  (HIGH → HIGH): {tp}"
)

print()

print(
    "Invalid model outputs:",
    len(invalid_predictions),
)

print(
    "Evaluation time:",
    f"{elapsed / 60:.2f} minutes",
)

print("=" * 70)

FINE-TUNED MODEL — HELD-OUT EVALUATION
Test transactions: 500

Processed 50/500 transactions (11.3s)
Processed 100/500 transactions (22.3s)
Processed 150/500 transactions (33.2s)
Processed 200/500 transactions (44.3s)
Processed 250/500 transactions (55.3s)
Processed 300/500 transactions (66.2s)
Processed 350/500 transactions (79.2s)
Processed 400/500 transactions (89.8s)
Processed 450/500 transactions (105.3s)
Processed 500/500 transactions (116.2s)

FINE-TUNED MODEL RESULTS
Accuracy:  0.4680 (46.80%)
Precision: 0.4796 (47.96%)
Recall:    0.7520 (75.20%)
F1 Score:  0.5857 (58.57%)

Confusion Matrix
----------------
True Negatives  (LOW  → LOW):  46
False Positives (LOW  → HIGH): 204
False Negatives (HIGH → LOW):  62
True Positives  (HIGH → HIGH): 188

Invalid model outputs: 0
Evaluation time: 1.94 minutes


In [46]:
from pathlib import Path

SAVE_DIR = Path(
    "/content/Financial-Fraud-Detection-System/"
    "artifacts/evaluation-data"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TEST_PATH = SAVE_DIR / "held_out_test_dataset"

test_dataset.save_to_disk(
    str(TEST_PATH)
)

print("=" * 70)
print("HELD-OUT TEST DATASET SAVED")
print("=" * 70)
print("Rows:", len(test_dataset))
print("Location:", TEST_PATH)
print("=" * 70)

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

HELD-OUT TEST DATASET SAVED
Rows: 500
Location: /content/Financial-Fraud-Detection-System/artifacts/evaluation-data/held_out_test_dataset


In [47]:
from huggingface_hub import HfApi

api = HfApi()

TEST_DATASET_REPO = (
    "ebbejulankapalli/"
    "financial-fraud-detector-heldout-test"
)

TEST_PATH = (
    "/content/Financial-Fraud-Detection-System/"
    "artifacts/evaluation-data/"
    "held_out_test_dataset"
)

# Create a PRIVATE dataset repository.
api.create_repo(
    repo_id=TEST_DATASET_REPO,
    repo_type="dataset",
    private=True,
    exist_ok=True,
)

api.upload_folder(
    repo_id=TEST_DATASET_REPO,
    repo_type="dataset",
    folder_path=TEST_PATH,
)

print("=" * 70)
print("HELD-OUT TEST DATASET BACKUP COMPLETE")
print("=" * 70)
print("Repository:", TEST_DATASET_REPO)
print("Rows: 500")
print("Visibility: PRIVATE")
print("Backup: SUCCESS")
print("=" * 70)

HELD-OUT TEST DATASET BACKUP COMPLETE
Repository: ebbejulankapalli/financial-fraud-detector-heldout-test
Rows: 500
Visibility: PRIVATE
Backup: SUCCESS


In [48]:
from huggingface_hub import HfApi

api = HfApi()

METRICS_FILE = (
    "/content/Financial-Fraud-Detection-System/"
    "evaluation/results/finetuned_model_metrics.json"
)

api.upload_file(
    path_or_fileobj=METRICS_FILE,
    path_in_repo="evaluation/finetuned_model_metrics.json",
    repo_id="ebbejulankapalli/financial-fraud-detector-qwen2.5-qlora",
    repo_type="model",
)

print("=" * 70)
print("EVALUATION METRICS BACKUP COMPLETE")
print("=" * 70)
print("Fine-tuned metrics: SAFE")
print("Adapter: SAFE")
print("Held-out test set: SAFE")
print("=" * 70)

EVALUATION METRICS BACKUP COMPLETE
Fine-tuned metrics: SAFE
Adapter: SAFE
Held-out test set: SAFE
